In [ ]:
#Devyani Deore - devyanid@umich.edu

**Load Libraries**

In [ ]:
import os
import re
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

**Download Dataset**

In [ ]:
import requests
import requests
request = requests.get("https://drive.google.com/uc?export=download&id=1wHt8PsMLsfX5yNSqrt2fSTcb8LEiclcf")
with open("data.zip", "wb") as file:
    file.write(request.content)

import zipfile
with zipfile.ZipFile('data.zip') as zip:
  zip.extractall('data')

**Load Train Data**

In [ ]:
data_complaint = pd.read_csv('data/complaint1700.csv')
data_complaint['label']=0
data_non_complaint = pd.read_csv('data/noncomplaint1700.csv')
data_non_complaint['label']=1
data = pd.concat([data_complaint, data_non_complaint], axis=0).reset_index(drop=True)
data.drop(['airline'], inplace=True, axis =1 )
data.sample(5)

In [ ]:
from sklearn.model_selection import train_test_split

x= data.tweet.values
y= data.label.values

x_train, x_val, y_train, y_val = train_test_split(x,y,test_size=0.2, random_state=2020)

**Load test data**


In [ ]:
test_data = pd.read_csv('data/test_data.csv')
test_data = test_data[['id','tweet']]
test_data.sample(5)

In [ ]:
import torch

if torch.cuda.is_available():
  device = torch.device("cuda")
  print(f'There are {torch.cuda.device_count()} GPU(s) available')
  print('Device name:', torch.cuda.get_device_name(0))

else:
  print('No GPU available, using the CPU instead.')
  device = torch.device("cpu")

**C-Baseline: TF-IDF + Naive Bayes Classifier**

**1. Data Preprocessing**


In [ ]:
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
import re # Ensure re module is imported if not already, though it was implicitly used before.

def text_preprocessing(s):
  s=s.lower()
  s=re.sub(r"\t","not",s)
  s = re.sub(r'(@.*?)\s', ' ', s)
  s = re.sub(r'([\'\"\.\(\)\!\?\\\/\,])', r' \1 ', s)
  s = re.sub(r'[^\w\s\?]', ' ', s)
  s = re.sub(r'([\;\:\|•«\n])', ' ', s)
  s = " ".join([word for word in s.split() if word not in stopwords.words('english')or word in ['not','can']])
  s = re.sub(r'\s+',' ',s).strip()
  return s

TF-IDF Vectorizer

In [ ]:
%%time
from sklearn.feature_extraction.text import TfidfVectorizer

# Preprocess text
X_train_preprocessed = np.array([text_preprocessing(text) for text in x_train])
X_val_preprocessed = np.array([text_preprocessing(text) for text in x_val])

# Calculate TF-IDF
tf_idf = TfidfVectorizer(ngram_range=(1, 3),
                         binary=True,
                         smooth_idf=False)
X_train_tfidf = tf_idf.fit_transform(X_train_preprocessed)
X_val_tfidf = tf_idf.transform(X_val_preprocessed)

**2. Train Naive Bayes Classifier**

2.1 Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

def get_auc_CV(model):
  kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
  auc = cross_val_score(model, X_train_tfidf, y_train, scoring="roc_auc", cv=kf)

  return auc.mean()

In [ ]:
from sklearn.naive_bayes import MultinomialNB
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

res = pd.Series([get_auc_CV(MultinomialNB(alpha=i))
                 for i in np.arange(1, 10, 0.1)],
                index=np.arange(1, 10, 0.1))

best_alpha = np.round(res.idxmax(), 2)
print('Best alpha: ', best_alpha)

plt.plot(res)
plt.title('AUC vs. Alpha')
plt.xlabel('Alpha')
plt.ylabel('AUC')
plt.show()

Evaluation on Validation Set

In [ ]:
from sklearn.metrics import accuracy_score, roc_curve, auc
def evaluate_roc(probs, y_true):

  preds = probs[:,1]
  fpr, tpr, thresholds = roc_curve(y_true, preds)
  roc_auc = auc(fpr, tpr)
  print(f'AUC: {roc_auc:.4f}')

  #get accuracy over test dataset
  y_pred = np.where(preds >= 0.5, 1, 0)
  accuracy = accuracy_score(y_true, y_pred)
  print(f'accuracy: {accuracy*100:.2f}%')

  plt.title('Receiver Operating Characteristic')
  plt.plot(fpr, tpr, 'b', label = 'AUC = %0.2f' % roc_auc)
  plt.legend(loc= 'lower right')
  plt.plot([0,1],[0,1], 'r--')
  plt.xlim([0,1])
  plt.ylim([0,1])
  plt.ylabel('True Positive Rate')
  plt.xlabel('False Positive Rate')
  plt.show()


In [ ]:
# Compute predicted probabilities
nb_model = MultinomialNB(alpha=1.8)
nb_model.fit(X_train_tfidf, y_train)
probs = nb_model.predict_proba(X_val_tfidf)

# Evaluate the classifier
evaluate_roc(probs, y_val)


**D - Fine-tuning**

1. Install the Hugging Face Library

In [ ]:
!pip install transformers

2. Tokenization and Input Formatting

In [ ]:
def text_preprocessing(text):
  text = re.sub(r'(@.*?)[\s]', ' ', text)
  text = re.sub(r'&amp;', '&', text)
  text = re.sub(r'\s+', ' ', text).strip()
  return text

In [ ]:
# Print sentence 0
print('Original: ', x[0])
print('Processed: ', text_preprocessing(x[0]))

2.1 BERT Tokenizer

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

def preprocessing_for_bert(data):
  input_ids=[]
  attention_masks = []

  for sent in data:

    encoded_sent = tokenizer( # Use tokenizer directly
        text_preprocessing(sent), # Pass the text directly
        add_special_tokens=True,
        max_length=MAX_LEN,
        padding='max_length', # Use 'padding' argument
        truncation=True,    # Add 'truncation' argument
        return_attention_mask=True
    )

    input_ids.append(encoded_sent.get('input_ids'))
    attention_masks.append(encoded_sent.get('attention_mask'))

  # Convert lists to tensors outside the loop
  input_ids = torch.tensor(input_ids)
  attention_masks = torch.tensor(attention_masks)

  return input_ids, attention_masks

In [ ]:
all_tweets = np.concatenate([data.tweet.values, test_data.tweet.values])

encoded_tweets = [tokenizer.encode(sent, add_special_tokens=True) for sent in all_tweets]

mex_len = max([len(sent) for sent in encoded_tweets])
print('Max length: ', mex_len)

In [ ]:
import torch # Add this import

MAX_LEN = 64

token_ids = list(preprocessing_for_bert([x[0]])[0].squeeze().numpy())
print('Original: ', x[0])
print('Token IDs: ', token_ids)

print('Tokenizing data...')
train_inputs, train_masks = preprocessing_for_bert(x_train)
val_inputs, val_masks = preprocessing_for_bert(x_val)

2.2 Create PyTorch DataLoader

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

train_labels = torch.tensor(y_train)
val_labels = torch.tensor(y_val)

batch_size=32

train_data = TensorDataset(train_inputs, train_masks, train_labels)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

val_data = TensorDataset(val_inputs, val_masks, val_labels)
val_sampler = SequentialSampler(val_data)
val_dataloader = DataLoader(val_data, sampler=val_sampler, batch_size=batch_size)

**3. Train Our Model**

Create BertClassifier

In [ ]:
%%time
import torch
import torch.nn as nn
from transformers import BertModel

class Bertclassifier(nn.Module):
  def __init__(self, freeze_bert=False):
    super(Bertclassifier, self).__init__()
    D_in, H, D_out = 768, 50, 2
    self.bert= BertModel.from_pretrained('bert-base-uncased')

    self.classifier = nn.Sequential(
        nn.Linear(D_in, H),
        nn.ReLU(),
        #nn.Dropout(0.5),
        nn.Linear(H, D_out)
    )

    if freeze_bert:
      for param in self.bert.parameters():
        param.requires_grad = False

  def forward(self, input_ids, attention_mask):
    outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

    last_hidden_state_cls = outputs[0][:,0,:]

    logits = self.classifier(last_hidden_state_cls)

    return logits

In [ ]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
import torch

# Re-define device to ensure it's available
if torch.cuda.is_available():
  device = torch.device("cuda")
  print(f'There are {torch.cuda.device_count()} GPU(s) available')
  print('Device name:', torch.cuda.get_device_name(0))
else:
  print('No GPU available, using the CPU instead.')
  device = torch.device("cpu")

def initialize_model(epochs=4):
  bert_classifier = Bertclassifier(freeze_bert=False)
  bert_classifier.to(device)
  optimizer = AdamW(bert_classifier.parameters(), lr=5e-5, eps=1e-8)
  return bert_classifier, optimizer

epochs = 4 # Define epochs in the global scope
bert_classifier, optimizer = initialize_model(epochs) # Call the function to get the optimizer
total_steps = len(train_dataloader) * epochs

scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

3.3 Training loop

In [ ]:
import random
import time

# Specify loss function
loss_fn = nn.CrossEntropyLoss()

def set_seed(seed_value=42):
    """Set seed for reproducibility.
    """
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)

def train(model, train_dataloader, val_dataloader=None, epochs=4, evaluation=False):
    """Train the BertClassifier model.
    """
    # Start training loop
    print("Start training...\n")
    for epoch_i in range(epochs):
        # =======================================
        #               Training
        # =======================================
        # Print the header of the result table
        print(f"{'Epoch':^7} | {'Batch':^7} | {'Train Loss':^12} | {'Val Loss':^10} | {'Val Acc':^9} | {'Elapsed':^9}")
        print("-"*70)

        # Measure the elapsed time of each epoch
        t0_epoch, t0_batch = time.time(), time.time()

        # Reset tracking variables at the beginning of each epoch
        total_loss, batch_loss, batch_counts = 0, 0, 0

        # Put the model into the training mode
        model.train()

        # For each batch of training data...
        for step, batch in enumerate(train_dataloader):
            batch_counts +=1
            # Load batch to GPU
            b_input_ids, b_attn_mask, b_labels = tuple(t.to(device) for t in batch)

            # Zero out any previously calculated gradients
            model.zero_grad()

            # Perform a forward pass. This will return logits.
            logits = model(b_input_ids, b_attn_mask)

            # Compute loss and accumulate the loss values
            loss = loss_fn(logits, b_labels)
            batch_loss += loss.item()
            total_loss += loss.item()

            # Perform a backward pass to calculate gradients
            loss.backward()

            # Clip the norm of the gradients to 1.0 to prevent "exploding gradients"
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            # Update parameters and the learning rate
            optimizer.step()
            scheduler.step()

            # Print the loss values and time elapsed for every 20 batches
            if (step % 20 == 0 and step != 0) or (step == len(train_dataloader) - 1):
                # Calculate time elapsed for 20 batches
                time_elapsed = time.time() - t0_batch

                # Print training results
                print(f"{epoch_i + 1:^7} | {step:^7} | {batch_loss / batch_counts:^12.6f} | {'-':^10} | {'-':^9} | {time_elapsed:^9.2f}")

                # Reset batch tracking variables
                batch_loss, batch_counts = 0, 0
                t0_batch = time.time()

        # Calculate the average loss over the entire training data
        avg_train_loss = total_loss / len(train_dataloader)

        print("-"*70)
        # =======================================
        #               Evaluation
        # =======================================
        if evaluation == True:
            # After the completion of each training epoch, measure the model's performance
            # on our validation set.
            val_loss, val_accuracy = evaluate(model, val_dataloader)

            # Print performance over the entire training data
            time_elapsed = time.time() - t0_epoch

            print(f"{epoch_i + 1:^7} | {'-':^7} | {avg_train_loss:^12.6f} | {val_loss:^10.6f} | {val_accuracy:^9.2f} | {time_elapsed:^9.2f}")
            print("-"*70)
        print("\n")

    print("Training complete!")


def evaluate(model, val_dataloader):
    """After the completion of each training epoch, measure the model's performance
    on our validation set.
    """
    # Put the model into the evaluation mode. The dropout layers are disabled during
    # the test time.
    model.eval()

    # Tracking variables
    val_accuracy = []
    val_loss = []

    # For each batch in our validation set...
    for batch in val_dataloader:
        # Load batch to GPU
        b_input_ids, b_attn_mask, b_labels = tuple(t.to(device) for t in batch)

        # Compute logits
        with torch.no_grad():
            logits = model(b_input_ids, b_attn_mask)

        # Compute loss
        loss = loss_fn(logits, b_labels)
        val_loss.append(loss.item())

        # Get the predictions
        preds = torch.argmax(logits, dim=1).flatten()

        # Calculate the accuracy rate
        accuracy = (preds == b_labels).cpu().numpy().mean() * 100
        val_accuracy.append(accuracy)

    # Compute the average accuracy and loss over the validation set.
    val_loss = np.mean(val_loss)
    val_accuracy = np.mean(val_accuracy)

    return val_loss, val_accuracy

In [ ]:
set_seed(42)

# Define the number of epochs for this training run
training_epochs = 2

# Call initialize_model to get the bert_classifier and optimizer
# It returns 2 values, so unpack only 2.
bert_classifier, optimizer = initialize_model(epochs=training_epochs)

# Calculate total_steps and create the scheduler separately
total_steps = len(train_dataloader) * training_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

train(bert_classifier, train_dataloader, val_dataloader, epochs=training_epochs, evaluation=True)

3.4 Evaluation on Validation Set

In [ ]:
import torch.nn.functional as F

def bert_predict(model, test_dataloader):
  model.eval()
  all_logits = []
  for batch in test_dataloader:
    b_input_ids, b_attn_mask = tuple(t.to(device) for t in batch)[:2]

    with torch.no_grad():
      logits = model(b_input_ids, b_attn_mask)
    all_logits.append(logits)

  all_logits = torch.cat(all_logits, dim=0)

  probs = F.softmax(all_logits, dim=1).cpu().numpy()

  return probs

In [ ]:
probs = bert_predict(bert_classifier, val_dataloader)
evaluate_roc(probs,y_val)

3.5 Train Model On Entire Training Dataset

In [ ]:
full_train_data = torch.utils.data.ConcatDataset([train_data, val_data])
full_train_sampler = RandomSampler(full_train_data)
full_train_dataloader = DataLoader(full_train_data, sampler=full_train_sampler, batch_size = 32)

set_seed(42)
# Call initialize_model to get the bert_classifier and optimizer
# It returns 2 values, so unpack only 2.
bert_classifier, optimizer = initialize_model(epochs=2)

# Calculate total_steps and create the scheduler separately
total_steps = len(full_train_dataloader) * 2 # 2 is the number of epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

train(bert_classifier, full_train_dataloader, epochs=2)

**4. Prediction on Test dataset**


4.1 Data Preparation

In [ ]:
test_data.sample(5)

In [ ]:
print('Tokenizing data....')
test_inputs, test_masks = preprocessing_for_bert(test_data.tweet)

test_dataset = TensorDataset(test_inputs, test_masks)
test_sampler = SequentialSampler(test_dataset)
test_dataloader = DataLoader(test_dataset, sampler=test_sampler, batch_size =32)

4.2 Predictions

In [ ]:
probs = bert_predict(bert_classifier, test_dataloader)
threshold = 0.9
preds = np.where(probs[:,1]> threshold, 1, 0)
print("Number of tweets predicted non-negative:", preds.sum())

In [ ]:
output = test_data[['id','tweet']]
print(list(output.sample()['tweet']))